In [1]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD


# ============================================================
# CONFIGURAÇÃO
# ============================================================

BASE_DIR = Path.home() / "Documentos" / "PUC" / "Projeto_Deep_Learning"

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "Embeddings" / "Not_CNN" / "Texto"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "validation", "test"]

N_COMPONENTS = 128

# Vocabulário máximo.
# Pode aumentar depois se quiser.
MAX_FEATURES = 30000

# Unigramas + bigramas
NGRAM_RANGE = (1, 2)


# ============================================================
# LOCALIZAR CSV
# ============================================================

def encontrar_csv(split):

    caminho = DATA_DIR / f"{split}.csv"

    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}"
        )

    return caminho


# ============================================================
# CARREGAR DADOS
# ============================================================

dfs = {}

for split in SPLITS:

    csv_path = encontrar_csv(split)

    print(f"\nCarregando {split}: {csv_path}")

    df = pd.read_csv(csv_path)

    colunas_necessarias = {"patch_id", "input"}

    if not colunas_necessarias.issubset(df.columns):
        raise ValueError(
            f"{csv_path} precisa conter as colunas "
            f"'patch_id' e 'input'. "
            f"Colunas encontradas: {list(df.columns)}"
        )

    df = df[["patch_id", "input"]].copy()

    df["input"] = df["input"].fillna("").astype(str)

    dfs[split] = df

    print(f"  Linhas: {len(df):,}")


# ============================================================
# TF-IDF
# ============================================================

print("\n" + "=" * 60)
print("AJUSTANDO TF-IDF SOMENTE NO TRAIN")
print("=" * 60)

vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    ngram_range=NGRAM_RANGE,
    sublinear_tf=True,
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(
    dfs["train"]["input"]
)

print(f"Dimensão TF-IDF: {X_train_tfidf.shape}")


# ============================================================
# SVD → 128 DIMENSÕES
# ============================================================

print("\n" + "=" * 60)
print("AJUSTANDO SVD → 128D")
print("=" * 60)

svd = TruncatedSVD(
    n_components=N_COMPONENTS,
    random_state=42
)

X_train = svd.fit_transform(X_train_tfidf)

print(
    f"Variância explicada pelos 128 componentes: "
    f"{svd.explained_variance_ratio_.sum():.4f}"
)


# ============================================================
# TRANSFORMAR E SALVAR
# ============================================================

for split in SPLITS:

    print(f"\nProcessando {split}...")

    X_tfidf = vectorizer.transform(
        dfs[split]["input"]
    )

    X_embedding = svd.transform(X_tfidf)

    result = pd.DataFrame(
        X_embedding,
        columns=[
            f"dim_{i:03d}"
            for i in range(N_COMPONENTS)
        ]
    )

    result.insert(
        0,
        "patch_id",
        dfs[split]["patch_id"].values
    )

    output_path = OUTPUT_DIR / f"{split}.parquet"

    result.to_parquet(
        output_path,
        index=False
    )

    print(
        f"  Salvo: {output_path}"
        f"\n  Shape: {result.shape}"
    )


print("\n" + "=" * 60)
print("PIPELINE DE TEXTO CONCLUÍDO")
print("=" * 60)


Carregando train: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/train.csv
  Linhas: 20,996

Carregando validation: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/validation.csv
  Linhas: 4,500

Carregando test: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/test.csv
  Linhas: 4,504

AJUSTANDO TF-IDF SOMENTE NO TRAIN
Dimensão TF-IDF: (20996, 208)

AJUSTANDO SVD → 128D
Variância explicada pelos 128 componentes: 1.0000

Processando train...
  Salvo: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/Embeddings/Not_CNN/Texto/train.parquet
  Shape: (20996, 129)

Processando validation...
  Salvo: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/Embeddings/Not_CNN/Texto/validation.parquet
  Shape: (4500, 129)

Processando test...
  Salvo: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/Embeddings/Not_CNN/Texto/test.parquet
  Shape: (4504, 129)

PIPELINE DE TEXTO CONCLUÍDO


In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio

from sklearn.decomposition import IncrementalPCA


# ============================================================
# CONFIGURAÇÃO
# ============================================================

BASE_DIR = Path.home() / "Documentos" / "PUC" / "Projeto_Deep_Learning"

DATA_DIR = BASE_DIR / "data"

# IMPORTANTE:
# Este diretório deve conter as bandas já redimensionadas
# para 96x96.
IMAGE_DIR = DATA_DIR / "BigEarthNet-S2-10bands"

OUTPUT_DIR = BASE_DIR / "Embeddings" / "Not_CNN" / "Imagem"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "validation", "test"]

BANDAS = [
    "B02",
    "B03",
    "B04",
    "B05",
    "B06",
    "B07",
    "B08",
    "B8A",
    "B11",
    "B12"
]

# Tamanho esperado de cada banda
TAMANHO_IMAGEM = 96

# Número final de dimensões do embedding
N_COMPONENTS = 128

# Quantidade de patches processados por vez
BATCH_SIZE = 256


# ============================================================
# LOCALIZAR CSV
# ============================================================

def encontrar_csv(split):

    caminho = DATA_DIR / f"{split}.csv"

    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}"
        )

    return caminho


# ============================================================
# CARREGAR IDS
# ============================================================

dfs = {}

for split in SPLITS:

    csv_path = encontrar_csv(split)

    print(f"\nCarregando {split}: {csv_path}")

    df = pd.read_csv(csv_path)

    if "patch_id" not in df.columns:
        raise ValueError(
            f"{csv_path} não possui a coluna 'patch_id'."
        )

    df = df[["patch_id"]].copy()

    dfs[split] = df

    print(f"  Patches: {len(df):,}")


# ============================================================
# LOCALIZAR PATCH
# ============================================================

def encontrar_patch(split, patch_id):

    caminho = IMAGE_DIR / split / patch_id

    if not caminho.exists():
        raise FileNotFoundError(
            f"Patch não encontrado:\n{caminho}"
        )

    return caminho


# ============================================================
# CARREGAR UM PATCH
# ============================================================

def carregar_patch(split, patch_id):

    patch_dir = encontrar_patch(split, patch_id)

    bandas = []

    for banda in BANDAS:

        arquivo = patch_dir / f"{patch_id}_{banda}.tif"

        if not arquivo.exists():
            raise FileNotFoundError(
                f"Banda não encontrada:\n{arquivo}"
            )

        with rasterio.open(arquivo) as src:

            imagem = src.read(1)

        if imagem.shape != (
            TAMANHO_IMAGEM,
            TAMANHO_IMAGEM
        ):
            raise ValueError(
                f"{arquivo.name} possui shape "
                f"{imagem.shape}, esperado "
                f"({TAMANHO_IMAGEM}, {TAMANHO_IMAGEM})."
            )

        bandas.append(imagem)

    # (10, 96, 96)
    imagem = np.stack(
        bandas,
        axis=0
    )

    # (92160,)
    imagem = imagem.reshape(-1)

    return imagem.astype(np.float32)


# ============================================================
# NÚMERO DE FEATURES
# ============================================================

N_FEATURES = (
    len(BANDAS)
    * TAMANHO_IMAGEM
    * TAMANHO_IMAGEM
)

print("\n" + "=" * 60)
print("CONFIGURAÇÃO DO EMBEDDING")
print("=" * 60)

print(f"Bandas: {len(BANDAS)}")
print(f"Tamanho: {TAMANHO_IMAGEM}x{TAMANHO_IMAGEM}")
print(f"Features por patch: {N_FEATURES:,}")
print(f"Embedding final: {N_COMPONENTS}D")
print(f"Batch size: {BATCH_SIZE}")


# ============================================================
# VALIDAÇÃO DO BATCH
# ============================================================

if BATCH_SIZE < N_COMPONENTS:

    raise ValueError(
        f"BATCH_SIZE ({BATCH_SIZE}) precisa ser "
        f">= N_COMPONENTS ({N_COMPONENTS}) "
        f"para o IncrementalPCA."
    )


# ============================================================
# ESTATÍSTICAS DO TRAIN
# ============================================================

print("\n" + "=" * 60)
print("CALCULANDO NORMALIZAÇÃO DO TRAIN")
print("=" * 60)

# Cada posição espacial de cada banda é uma feature.
#
# Agora temos:
#
# 10 × 96 × 96 = 92.160 features

soma = np.zeros(
    N_FEATURES,
    dtype=np.float64
)

soma_quadrado = np.zeros(
    N_FEATURES,
    dtype=np.float64
)

n_train = len(dfs["train"])

for inicio in range(
    0,
    n_train,
    BATCH_SIZE
):

    fim = min(
        inicio + BATCH_SIZE,
        n_train
    )

    batch_ids = dfs["train"]["patch_id"].iloc[
        inicio:fim
    ]

    X_batch = np.stack([
        carregar_patch(
            "train",
            patch_id
        )
        for patch_id in batch_ids
    ])

    soma += X_batch.sum(
        axis=0
    )

    soma_quadrado += (
        X_batch ** 2
    ).sum(axis=0)

    print(
        f"\rEstatísticas: "
        f"{fim:,}/{n_train:,}",
        end=""
    )

print()


# ============================================================
# MÉDIA E DESVIO
# ============================================================

media = soma / n_train

variancia = (
    soma_quadrado / n_train
    - media ** 2
)

variancia = np.maximum(
    variancia,
    0
)

desvio = np.sqrt(
    variancia
)

# Evita divisão por zero
desvio[
    desvio < 1e-8
] = 1.0


# ============================================================
# INCREMENTAL PCA
# ============================================================

print("\n" + "=" * 60)
print("AJUSTANDO INCREMENTAL PCA → 128D")
print("=" * 60)

ipca = IncrementalPCA(
    n_components=N_COMPONENTS,
    batch_size=BATCH_SIZE
)


# ============================================================
# FIT SOMENTE NO TRAIN
# ============================================================

for inicio in range(
    0,
    n_train,
    BATCH_SIZE
):

    fim = min(
        inicio + BATCH_SIZE,
        n_train
    )

    batch_ids = dfs["train"]["patch_id"].iloc[
        inicio:fim
    ]

    # --------------------------------------------------------
    # Proteção contra último batch < 128
    # --------------------------------------------------------

    if fim - inicio < N_COMPONENTS:

        print(
            f"\nÚltimo batch possui "
            f"{fim - inicio} amostras."
        )

        print(
            "Esse batch será ignorado no "
            "ajuste do PCA."
        )

        break

    X_batch = np.stack([
        carregar_patch(
            "train",
            patch_id
        )
        for patch_id in batch_ids
    ])

    # --------------------------------------------------------
    # Normalização usando somente estatísticas do train
    # --------------------------------------------------------

    X_batch = (
        X_batch - media
    ) / desvio

    # --------------------------------------------------------
    # Atualiza PCA
    # --------------------------------------------------------

    ipca.partial_fit(
        X_batch
    )

    print(
        f"\rPCA: "
        f"{fim:,}/{n_train:,}",
        end=""
    )

print()

print(
    f"Componentes aprendidos: "
    f"{ipca.components_.shape}"
)


# ============================================================
# TRANSFORMAR E SALVAR
# ============================================================

for split in SPLITS:

    print("\n" + "=" * 60)
    print(
        f"GERANDO EMBEDDINGS: "
        f"{split.upper()}"
    )
    print("=" * 60)

    df = dfs[split]

    embeddings = []

    for inicio in range(
        0,
        len(df),
        BATCH_SIZE
    ):

        fim = min(
            inicio + BATCH_SIZE,
            len(df)
        )

        batch_ids = df["patch_id"].iloc[
            inicio:fim
        ]

        X_batch = np.stack([
            carregar_patch(
                split,
                patch_id
            )
            for patch_id in batch_ids
        ])

        # ----------------------------------------------------
        # Normalização usando SOMENTE o train
        # ----------------------------------------------------

        X_batch = (
            X_batch - media
        ) / desvio

        # ----------------------------------------------------
        # PCA
        # ----------------------------------------------------

        X_embedding = ipca.transform(
            X_batch
        )

        embeddings.append(
            X_embedding.astype(
                np.float32
            )
        )

        print(
            f"\r{fim:,}/{len(df):,}",
            end=""
        )

    print()

    # Junta os batches
    X_embedding = np.vstack(
        embeddings
    )

    # --------------------------------------------------------
    # DataFrame final
    # --------------------------------------------------------

    result = pd.DataFrame(
        X_embedding,
        columns=[
            f"dim_{i:03d}"
            for i in range(
                N_COMPONENTS
            )
        ]
    )

    result.insert(
        0,
        "patch_id",
        df["patch_id"].values
    )

    # --------------------------------------------------------
    # Salvar
    # --------------------------------------------------------

    output_path = (
        OUTPUT_DIR
        / f"{split}.parquet"
    )

    result.to_parquet(
        output_path,
        index=False
    )

    print(
        f"Salvo: {output_path}"
        f"\nShape: {result.shape}"
    )


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 60)
print("PIPELINE DE IMAGEM — NOT_CNN — CONCLUÍDO")
print("=" * 60)

print(
    f"Entrada: "
    f"{len(BANDAS)} × "
    f"{TAMANHO_IMAGEM} × "
    f"{TAMANHO_IMAGEM}"
)

print(
    f"Features: {N_FEATURES:,}"
)

print(
    f"Saída: {N_COMPONENTS}D"
)

print(
    f"Diretório: {OUTPUT_DIR}"
)


Carregando train: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/train.csv
  Patches: 20,996

Carregando validation: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/validation.csv
  Patches: 4,500

Carregando test: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/test.csv
  Patches: 4,504

CONFIGURAÇÃO DO EMBEDDING
Bandas: 10
Tamanho: 96x96
Features por patch: 92,160
Embedding final: 128D
Batch size: 256

CALCULANDO NORMALIZAÇÃO DO TRAIN
Estatísticas: 20,996/20,996

AJUSTANDO INCREMENTAL PCA → 128D
PCA: 20,992/20,996
Último batch possui 4 amostras.
Esse batch será ignorado no ajuste do PCA.

Componentes aprendidos: (128, 92160)

GERANDO EMBEDDINGS: TRAIN
20,996/20,996
Salvo: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/Embeddings/Not_CNN/Imagem/train.parquet
Shape: (20996, 129)

GERANDO EMBEDDINGS: VALIDATION
4,500/4,500
Salvo: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/Embeddings/Not_CNN/Imagem/validation.parquet
Shape: (4500, 129)

GERANDO EMBEDDINGS